In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("catalog_name", "catalog_smartdata")
dbutils.widgets.text("storage_account", "adlsjrdata")
dbutils.widgets.text("storage_credential_name", "credential")

In [0]:
dbutils.widgets.text("raw_container", "raw")
dbutils.widgets.text("bronze_container", "bronze")
dbutils.widgets.text("silver_container", "silver")
dbutils.widgets.text("gold_container", "gold")

# External Locations
dbutils.widgets.text("external_location_raw", "exlt-raw")
dbutils.widgets.text("external_location_bronze", "exlt-bronze")
dbutils.widgets.text("external_location_silver", "exlt-silver")
dbutils.widgets.text("external_location_gold", "exlt-gold")


In [0]:
current_user = spark.sql("SELECT current_user()").collect()[0][0]

USO MI CURRENT USER PARA LOS GRANTS PORQUE NO PUEDO ACCEDER AL USER MANAGEMENT

In [0]:
read_principal = current_user
write_principal = current_user

In [0]:
catalog_name = dbutils.widgets.get("catalog_name")
storage_account = dbutils.widgets.get("storage_account")
storage_credential_name = dbutils.widgets.get("storage_credential_name")

raw_container = dbutils.widgets.get("raw_container")
bronze_container = dbutils.widgets.get("bronze_container")
silver_container = dbutils.widgets.get("silver_container")
gold_container = dbutils.widgets.get("gold_container")

external_location_raw = dbutils.widgets.get("external_location_raw")
external_location_bronze = dbutils.widgets.get("external_location_bronze")
external_location_silver = dbutils.widgets.get("external_location_silver")
external_location_gold = dbutils.widgets.get("external_location_gold")


In [0]:
spark.sql(f"USE CATALOG {catalog_name}")

In [0]:
locations = [
    {
        "name": external_location_raw,
        "url": f"abfss://{raw_container}@{storage_account}.dfs.core.windows.net/",
        "comment": "Ubicación externa para datos RAW"
    },
    {
        "name": external_location_bronze,
        "url": f"abfss://{bronze_container}@{storage_account}.dfs.core.windows.net/",
        "comment": "Ubicación externa para capa BRONZE"
    },
    {
        "name": external_location_silver,
        "url": f"abfss://{silver_container}@{storage_account}.dfs.core.windows.net/",
        "comment": "Ubicación externa para capa SILVER"
    },
    {
        "name": external_location_gold,
        "url": f"abfss://{gold_container}@{storage_account}.dfs.core.windows.net/",
        "comment": "Ubicación externa para capa GOLD"
    },
]

In [0]:
for loc in locations:
    try:
        dbutils.fs.ls(loc["url"])
        print(f" Acceso OK: {loc['url']}")
    except Exception as e:
        print(f" Error acceso: {loc['url']}")
        raise e


In [0]:
for loc in locations:
    stmt = f"""
    CREATE EXTERNAL LOCATION IF NOT EXISTS `{loc["name"]}`
    URL '{loc["url"]}'
    WITH (STORAGE CREDENTIAL `{storage_credential_name}`)
    COMMENT '{loc["comment"]}'
    """
    print("Ejecutando:")
    print(stmt.strip())
    spark.sql(stmt)

print("External locations creados correctamente.")

GRANTS - RAW - BRONZE - SILVER - GOLD

In [0]:
grant_statements = [

    # RAW (solo lectura)
    f"GRANT READ FILES ON EXTERNAL LOCATION `{external_location_raw}` TO `{read_principal}`",
    f"GRANT READ FILES ON EXTERNAL LOCATION `{external_location_raw}` TO `{write_principal}`",

    # BRONZE
    f"GRANT READ FILES, WRITE FILES ON EXTERNAL LOCATION `{external_location_bronze}` TO `{write_principal}`",

    # SILVER
    f"GRANT READ FILES, WRITE FILES ON EXTERNAL LOCATION `{external_location_silver}` TO `{write_principal}`",

    # GOLD
    f"GRANT READ FILES, WRITE FILES ON EXTERNAL LOCATION `{external_location_gold}` TO `{write_principal}`",
]